# DATA 622: Homework 7
**Author:** Brett Allen (ballen3@umbc.edu)

**Date Completed:** TBD

## Setup

In [1]:
!python -m pip install -q requests beautifulsoup4 transformers scikit-learn nltk textblob

### Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import re
import nltk
import numpy as np
from transformers import pipeline
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

/home/brett/anaconda3/envs/data-science/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configurations

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/brett/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/brett/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Questions
Extract the article of Sam Altman’s interview https://venturebeat.com/ai/sam-altman-at-ted-2025-inside-the-most-uncomfortable-and-important-ai-interview-of-the-year/.

In [4]:
url = "https://venturebeat.com/ai/sam-altman-at-ted-2025-inside-the-most-uncomfortable-and-important-ai-interview-of-the-year/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)
response.status_code

200

In [5]:
# Parse with beautiful soup
soup = BeautifulSoup(response.text, "html.parser")

# Extract all paragraph text
paragraphs = soup.find_all("p")
article_text = " ".join([p.get_text() for p in paragraphs])

# Clean text
article_text = re.sub(r'\s+', ' ', article_text)

# Preview first 1000 characters
print(article_text[:1000] + '...') 

OpenAI CEO Sam Altman revealed that his company has grown to 800 million weekly active users and is experiencing "unbelievable" growth rates, during a sometimes tense interview at the TED 2025 conference in Vancouver last week. "I have never seen growth in any company, one that I've been involved with or not, like this," Altman told TED head Chris Anderson during their on-stage conversation. "The growth of ChatGPT — it is really fun. I feel deeply honored. But it is crazy to live through, and our teams are exhausted and stressed." The interview, which closed out the final day of TED 2025: Humanity Reimagined, showcased not just OpenAI's skyrocketing success but also the increasing scrutiny the company faces as its technology transforms society at a pace that alarms even some of its supporters. Altman painted a picture of a company struggling to keep up with its own success, noting that OpenAI's GPUs are "melting" due to the popularity of its new image generation features. "All day long

### 1. Use an LLM to summarize the article.

In [7]:
# Initialize LLM
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 338/338 [00:01<00:00, 174.74it/s]


In [9]:
prompt = f"""
Provide a summary for the following article. Only provide the summary and nothing else.

Article:
{article_text}
"""

result = llm(
    prompt,
    max_length=2048,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.3
)

summary = result[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [14]:
def substr_after(text: str, after: str) -> str:
    if after in text:
        idx = text.index(after)
        return text[idx+len(after):].strip()
    return text

In [15]:
# Ensure the prompt isn't included in the response
summary = substr_after(text=summary, after=prompt)

In [16]:
# Ensure "Summary" title isn't in the summary
summary = substr_after(text=summary, after="Summary:")

In [17]:
print("LLM Generated Summary:\n")
print(summary)

LLM Generated Summary:

This article discusses the rapid growth of OpenAI, a leading AI research company, and the ethical dilemmas surrounding its development. OpenAI has experienced tremendous growth since its inception, reaching 800 million weekly active users and facing intense scrutiny amid its rapidly expanding influence. Despite the company's efforts to maintain control, the sheer volume of users poses significant challenges, such as managing GPU resources and addressing potential risks associated with autonomous AI agents. OpenAI has also faced criticism for its handling of intellectual property issues, particularly concerning AI-generated artwork. The company's CEO, Sam Altman, acknowledges the immense power he holds but maintains that OpenAI continues to prioritize making AGI accessible and beneficial for humanity. Additionally, OpenAI is exploring various policy changes, including loosening restrictions on image generation models and shifting towards allowing users greater au

### 2. Identify the key topics, and the sentiment. Is the sentiment measured by the LLM different from one using a classification model such as Naïve Bayes or Support Vector Machine?

#### Key Topics (using LLM)

In [22]:
key_topics_prompt = f"""
List the key topics for the following article. Only provide the key topics and nothing else.

Article:
{article_text}
"""

result = llm(
    key_topics_prompt,
    max_length=2048,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.3
)

key_topics = result[0]["generated_text"]

Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [23]:
# Ensure the prompt isn't included in the response
key_topics = substr_after(text=key_topics, after=key_topics_prompt)

In [25]:
# Ensure "Key Topics" title isn't in the summary
key_topics = substr_after(text=key_topics, after="Key Topics:")

In [26]:
print('LLM Generated Key Topics:\n')
print(key_topics)

LLM Generated Key Topics:

- OpenAI's growth and success
- Unprecedented growth rates
- ChatGPT and its impact
- Exponential growth challenges
- Social media competition
- Valuation and funding
- Policy changes
- Autonomous agents and AI safety
- Content moderation shifts
- Future of creativity and AI
- Copyright and intellectual property
- Safety and ethical considerations
- Corporate governance and decision-making
- Stakeholder engagement and criticism

These topics cover the main themes discussed in the article, including the rapid expansion of OpenAI, technological advancements, regulatory issues, and broader implications for society and the future of AI.


#### Sentiment Classification (using LLM)

In [35]:
sentiment_prompt = f"""
Analyze the sentiment of the following article.

Return ONLY one word: Positive, Negative, or Neutral.

Article:
{article_text}
"""

result = llm(
    sentiment_prompt,
    max_new_tokens=1,
    do_sample=False
)

llm_sentiment = result[0]["generated_text"]

Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [36]:
# Ensure the prompt isn't included in the response
llm_sentiment = substr_after(text=llm_sentiment, after=sentiment_prompt)

In [38]:
print('LLM Sentiment:\n')
print(llm_sentiment)

LLM Sentiment:

Neutral


#### Sentiment using Traditional ML

##### Obtain Sentiment Training Dataset (from Kaggle)
Dataset source: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset

##### Train Naive Bayes Model

##### Sentiment Classification (Naive Bayes)

##### Train SVM

##### Sentiment Classification (SVM)

### 3. What is the general emotion of the article?

### 4. What is the main theme of the article?